In [ ]:
// load the formatting setup, the visualization functions, and the style sheet
#load "../setup.fsx"
#load "../visuals.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# Using an External SAT Solver

The Python version of this notebook uses [PicoSat](https://fmv.jku.at/picosat/), which is a SAT solver by [Armin Biere](https://cca.informatik.uni-freiburg.de/biere/).  As there is no F# interface to PicoSat, we use [CaDiCaL](https://github.com/arminbiere/cadical) instead.  CaDiCaL is a more recent SAT solver by the same author.  It can be installed into the anaconda environment `fsharp` via
```
conda install -n fsharp -c conda-forge cadical
```
CaDiCaL is a command line program.  It reads a formula in conjunctive normal form from a file that is written in the *DIMACS* format.  In this format, propositional variables are represented as follows:
* A propositional variable $p$ is represented as a positive natural number.
* If $n$ is the natural number representing the propositional variable $p$, then $\neg p$ is represented as the integer $-n$.
* A clause is represented as a line containing the integers representing its literals.  The line is terminated by the number `0`.
* The first line of the file has the form `p cnf v c` where `v` is the number of variables and `c` is the number of clauses.

For example, if the propositional variables $p$, $q$, and $r$ are represented as the natural numbers
$1$, $2$, and $3$ respectively, then the formula 
$$ (p \vee \neg q \vee r) \wedge (\neg p \vee q \vee \neg r) \wedge (\neg p \vee \neg q \vee r) \wedge (p \vee q \vee \neg r), $$
is represented as follows:
```
p cnf 3 4
1 -2 3 0
-1 2 -3 0
-1 -2 3 0
1 2 -3 0
```
In F#, we represent a formula in this format as a list of lists of integers.

In [ ]:
let f = [ [1; -2; 3]; [-1; 2; -3]; [-1; -2; 3]; [1; 2; -3] ]

## Calling the SAT Solver

The function `toDimacs` takes one argument:
- `clauses` is a list of clauses, where every clause is a list of non-zero integers.

It returns a string containing these clauses in DIMACS format.  The number of variables is the largest absolute value of a literal.

In [ ]:
let toDimacs (clauses: int list list) : string =
    let numVars = clauses |> List.concat |> List.map abs |> List.fold max 0
    let lines   = [ for clause in clauses ->
                      String.concat " " [ for l in clause -> string l ] + " 0" ]
    sprintf "p cnf %d %d\n" numVars (List.length clauses) + String.concat "\n" lines + "\n"

In [ ]:
printfn "%s" (toDimacs f)

The function `satSolve` takes one argument:
- `clauses` is a list of clauses in the format described above.

It calls the SAT solver CaDiCaL to check whether `clauses` is satisfiable.  The result has the type `int list option`:
- If the clauses are satisfiable, `Some literals` is returned.  Here, `literals` is a list of integers that contains for every variable $n$ either $n$ (if the variable is true) or $-n$ (if the variable is false).
- If the clauses are unsatisfiable, `None` is returned.

**Implementation:** 
1. The clauses are written into a temporary file in DIMACS format.
2. The program `cadical` is started with the options `-q` (quiet) and the name of the file.  The output of the program is read into the string `output`.  
3. CaDiCaL prints a line starting with `s` that is either `s SATISFIABLE` or `s UNSATISFIABLE`.  In the first case, the solution is printed on lines starting with `v`.  These lines contain the literals that are true, terminated by `0`.  We collect all these numbers except for the final `0`.

The program `cadical` is searched in the directories listed in the environment variable `PATH`.  If the notebook is started from the activated anaconda environment `fsharp`, the directory containing `cadical` is part of `PATH`.  Otherwise, we try the directory `bin` of the environment given by the variable `CONDA_PREFIX`.

In [ ]:
open System.Diagnostics

let cadical : string =
    let prefix = System.Environment.GetEnvironmentVariable "CONDA_PREFIX"
    let candidate = if isNull prefix then "" else System.IO.Path.Combine(prefix, "bin", "cadical")
    if System.IO.File.Exists candidate then candidate else "cadical"

let satSolve (clauses: int list list) : int list option =
    let file = System.IO.Path.GetTempFileName()
    System.IO.File.WriteAllText(file, toDimacs clauses)
    let info = ProcessStartInfo(cadical, sprintf "-q \"%s\"" file)
    info.RedirectStandardOutput <- true
    use proc   = Process.Start info
    let output = proc.StandardOutput.ReadToEnd()
    proc.WaitForExit()
    System.IO.File.Delete file
    let lines = output.Split('\n') |> Array.map (fun line -> line.Trim())
    if Array.contains "s SATISFIABLE" lines then
        Some [ for line in lines do
                 if line.StartsWith "v" then
                     for token in line.Substring(1).Split(' ', System.StringSplitOptions.RemoveEmptyEntries) do
                         if token <> "0" then int token ]
    else
        None

Let us check whether the formula `f` is satisfiable:

In [ ]:
satSolve f

This shows that the formula `f` is satisfiable.  For example, the propositional valuation 
$$ \mathcal{I} = \{ p \mapsto \texttt{false}, q \mapsto \texttt{false}, r \mapsto \texttt{false} \} $$
is a solution for `f`, i.e. we have
$$ \mathcal{I}(\texttt{f}) = \texttt{true}. $$
The solution found by CaDiCaL might be different from the solution found by PicoSat, since `f` has more than one solution.

The formula $p \wedge \neg p$ is unsatisfiable:

In [ ]:
satSolve [ [1]; [-1] ]

## Transforming Clauses into DIMACS Format

In order to use the SAT solver for the examples discussed in our lecture, we need a function that transforms a set of clauses
into the integer format.  Furthermore, we need a function that can translate a solution found by the SAT solver back into our format.  We use the types `Literal`, `Clause`, and `CNF` that are defined in the notebook `04-CNF.ipynb`.

In [ ]:
importNotebook "04-CNF.ipynb"

The function `findVariables` takes one argument:
- `Clauses` is a set of clauses.

It returns the set of all propositional variables occurring in `Clauses`.

In [ ]:
let findVariables (Clauses: CNF) : Set<string> =
    set [ for C in Clauses do
            for l in C do
                match l with
                | Pos p -> p
                | Neg p -> p ]

The function `numberVariables` takes one argument:
- `Clauses` is a set of clauses.

It returns a pair of two maps:
* The map `Var2Int` maps every propositional variable occurring in `Clauses` to a unique natural number.  The variables are numbered starting with $1$.
* The map `Int2Var` is the mapping that is inverse to the map `Var2Int`.

`List.indexed` turns a list `[x0; x1; ...]` into the list of pairs `[(0, x0); (1, x1); ...]`.

In [ ]:
let numberVariables (Clauses: CNF) : Map<string, int> * Map<int, string> =
    let numbered = [ for (i, p) in List.indexed (Set.toList (findVariables Clauses)) -> (p, i + 1) ]
    let Var2Int  = Map numbered
    let Int2Var  = Map [ for (p, n) in numbered -> (n, p) ]
    (Var2Int, Int2Var)

The function `literalToInt` takes two arguments:
- `l` is a literal,
- `Var2Int` is a map assigning natural numbers to propositional variables.

It returns the integer representing `l`.  If `l` is a negated variable, the integer is negative, else it is positive.

In [ ]:
let literalToInt (l: Literal) (Var2Int: Map<string, int>) : int =
    match l with
    | Pos p ->  Var2Int[p]
    | Neg p -> -Var2Int[p]

The function `clausesToInts` takes two arguments:
- `Clauses` is a set of clauses,
- `Var2Int` is a map assigning natural numbers to propositional variables.

It transforms `Clauses` into a list of lists of integers.

In [ ]:
let clausesToInts (Clauses: CNF) (Var2Int: Map<string, int>) : int list list =
    [ for C in Clauses -> [ for l in C -> literalToInt l Var2Int ] ]

The function `intsToUnitClauses` takes two arguments:
- `Numbers` is a list of integers representing literals,
- `Int2Var` is a map assigning propositional variables to natural numbers.

It returns the set of unit clauses corresponding to these literals.

In [ ]:
let intsToUnitClauses (Numbers: int list) (Int2Var: Map<int, string>) : CNF =
    set [ for n in Numbers ->
            if n > 0 then set [ Pos Int2Var[n] ] else set [ Neg Int2Var[-n] ] ]

## Sudoku

As an example, we solve a Sudoku.  The Finnish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is defined below.  The puzzle is represented as an array of rows, where every row is an array of digits.  Empty fields are represented by the digit `0`.  The syntax `[| ... |]` creates an array.

In [ ]:
let puzzle : int[][] =
    [| [| 8; 0; 0; 0; 0; 0; 0; 0; 0 |]
       [| 0; 0; 3; 6; 0; 0; 0; 0; 0 |]
       [| 0; 7; 0; 0; 9; 0; 2; 0; 0 |]
       [| 0; 5; 0; 0; 0; 7; 0; 0; 0 |]
       [| 0; 0; 0; 0; 4; 5; 7; 0; 0 |]
       [| 0; 0; 0; 1; 0; 0; 0; 3; 0 |]
       [| 0; 0; 1; 0; 0; 0; 0; 6; 8 |]
       [| 0; 0; 8; 5; 0; 0; 0; 1; 0 |]
       [| 0; 9; 0; 0; 0; 0; 4; 0; 0 |] |]

We will solve this Sudoku by coding it as a set of clauses.  We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `var` takes three arguments:
- `row` is the number of a row,
- `col` is the number of a column,
- `digit` is a digit.

It returns the string `Q<row,col,digit>`, which is interpreted as a variable name.

In [ ]:
let var (row: int) (col: int) (digit: int) : string =
    sprintf "Q<%d,%d,%d>" row col digit

In [ ]:
var 1 2 3

The function `atMostOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at most one of the variables of `S` is true.

In [ ]:
let atMostOne (S: Set<string>) : CNF =
    set [ for p in S do
            for q in S do
                if p < q then
                    set [ Neg p; Neg q ] ]

The function `atLeastOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at least one of the variables of `S` is true.  This set contains just one clause.

In [ ]:
let atLeastOne (S: Set<string>) : CNF =
    set [ Set.map Pos S ]

The function `exactlyOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that exactly one of the variables of `S` is true.

In [ ]:
let exactlyOne (S: Set<string>) : CNF =
    atMostOne S + atLeastOne S

In [ ]:
exactlyOne (set [ "a"; "b"; "c" ])

The function `allDifferent` takes one argument:
- `L` is a list of pairs of the form `(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.  `L` contains nine pairs.

It returns a set of clauses expressing that all Sudoku fields specified by the coordinate pairs in `L` take different digits as values.  As there are nine fields, this is the case if every digit occurs exactly once in these fields.

In [ ]:
let allDifferent (L: (int * int) list) : CNF =
    Set.unionMany [ for digit in 1 .. 9 ->
                      exactlyOne (set [ for (row, col) in L -> var row col digit ]) ]

In [ ]:
allDifferent [ for c in 1 .. 9 -> (1, c) ]

The function `exactlyOneDigit` takes two arguments:
- `row` is the number of a row,
- `col` is the number of a column.

The function returns a set of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `true`.

In [ ]:
let exactlyOneDigit (row: int) (col: int) : CNF =
    exactlyOne (set [ for digit in 1 .. 9 -> var row col digit ])

In [ ]:
exactlyOneDigit 1 1

The function `constraintsFromPuzzle` takes no arguments.  It returns a set of unit clauses stating that the variables corresponding to digits that are already given in the Sudoku puzzle take the values that are specified.  Note that the arrays are indexed starting from `0`, while the rows and columns of the Sudoku are numbered starting from `1`.

In [ ]:
let constraintsFromPuzzle () : CNF =
    set [ for row in 0 .. 8 do
            for col in 0 .. 8 do
                if puzzle[row][col] <> 0 then
                    set [ Pos (var (row + 1) (col + 1) (puzzle[row][col])) ] ]

In [ ]:
constraintsFromPuzzle ()

The function `allConstraints` takes no arguments.  It returns a set of clauses that encodes the given Sudoku.  The clauses express that
- the digits given in the puzzle are respected,
- there is exactly one digit in every field,
- all entries in a row are different,
- all entries in a column are different, and
- all entries in each of the nine $3 \times 3$ squares are different.

In [ ]:
let allConstraints () : CNF =
    let L = [ 1 .. 9 ]
    // the constraints from the puzzle have to be satisfied
    let mutable Clauses = constraintsFromPuzzle ()
    // there is exactly one digit in every field
    for row in L do
        for col in L do
            Clauses <- Clauses + exactlyOneDigit row col
    // all entries in a row are unique
    for row in L do
        Clauses <- Clauses + allDifferent [ for col in L -> (row, col) ]
    // all entries in a column are unique
    for col in L do
        Clauses <- Clauses + allDifferent [ for row in L -> (row, col) ]
    // all entries in a 3x3 square are unique
    for r in 0 .. 2 do
        for c in 0 .. 2 do
            Clauses <- Clauses + allDifferent [ for row in 1 .. 3 do
                                                  for col in 1 .. 3 -> (r * 3 + row, c * 3 + col) ]
    Clauses

We print the unit clauses and the clauses of length 9.

In [ ]:
let Constraints = allConstraints ()
for C in Constraints do
    if C.Count = 1 then
        printfn "%s" (clauseToString C)
for C in Constraints do
    if C.Count = 9 then
        printfn "%s" (clauseToString C)

In [ ]:
Constraints.Count

The function `sudoku` takes no arguments.  It solves the Sudoku using the SAT solver CaDiCaL.  The result has the type `CNF option`:
- If the Sudoku is solvable, `Some Solution` is returned, where `Solution` is a set of unit clauses that specifies the values of all variables.
- Otherwise, a message is printed and `None` is returned.

In [ ]:
let sudoku () : CNF option =
    let (V2I, I2V) = numberVariables Constraints
    match satSolve (clausesToInts Constraints V2I) with
    | Some literals -> Some (intsToUnitClauses literals I2V)
    | None          -> printfn "The problem is not solvable!"
                       None

Even though this Sudoku is modelled using $9^3 = 729$ propositional variables and more than 10000 clauses, the SAT solver needs only a few milliseconds to solve the problem.  Most of the time is spent starting the program and transforming the clauses.

In [ ]:
#!time
let Solution = sudoku ()

## Graphical Representation

The function `extractRowColDigit` takes one argument:
- `name` is a variable of the form `Q<row,col,digit>`.

It returns the triple `(row, col, digit)`.

**Implementation:** We remove the first two characters `Q<` and the last character `>`.  The remaining string is split at the commas and the three parts are converted to integers.

In [ ]:
let extractRowColDigit (name: string) : int * int * int =
    let parts = name.Substring(2, name.Length - 3).Split(',')
    (int parts[0], int parts[1], int parts[2])

The function `transformSolution` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the Sudoku.

It returns a map that assigns the digit to every position `(row, col)`.  Only the positive literals of `Solution` are relevant.

In [ ]:
let transformSolution (Solution: CNF) : Map<int * int, int> =
    Map [ for C in Solution do
            for l in C do
                match l with
                | Pos p -> let (row, col, digit) = extractRowColDigit p
                           ((row, col), digit)
                | Neg _ -> () ]

The function `showSolution` takes two arguments:
- `Solution` is a set of unit clauses representing a solution of the Sudoku,
- `width` is the width of the grid as a CSS length.

It displays the Sudoku.  The digits that were given in the puzzle are shown in black, the digits computed by the solver are shown in blue.  The function `sudokuGrid` is defined in the file `visuals.fsx`, which has been loaded in the first cell of this notebook.

In [ ]:
let showSolution (Solution: CNF) (width: string) =
    sudokuGrid puzzle (transformSolution Solution) width

In [ ]:
showSolution Solution.Value "50%"